| 维度 | 大基因组 (如人类/玉米) | 小基因组 (T. majus) |
| :--- | :--- | :--- |
| **STAR 内存需求** | 30 - 40 GB | **2.5 - 4 GB** |
| **并行策略** | 串行或限制 maxForks=2 | **并行 maxForks=8+** |
| **运行预估时间** | 12 - 24 小时 | **1.5 - 3 小时** |
| **磁盘占用** | 巨大 | **极小** |

### 项目运行环境设置 (Persistent Settings)

| 维度 | 设置参数 | 专家理由 |
| :--- | :--- | :--- |
| **会话管理** | **Tmux (Session: `rnaseq_majus`)** | 防止 SSH 断线导致 Nextflow 主进程中断及 Socket 泄露。 |
| **并行策略** | **High Parallel (Forks: 8)** | *T. majus* 基因组极小 (232Mb)，支持 8 个样本同时比对。 |
| **内存阈值** | **Max RAM: 85GB** | 预留 ~10GB 给系统及子进程，防止触发 OOM Killer。 |
| **核心分配** | **32 Threads (4 per sample)** | 达到 CPU 饱和状态，且避免单样本过度消耗导致上下文切换。 |
| **持久化开关** | **-resume** | 确保在任务意外中断后，利用磁盘缓存瞬间恢复进度。 |

### *Tropaeolum majus* RNA-seq 运行设置汇总

| 配置项 | 设定值 | 状态说明 |
| :--- | :--- | :--- |
| **会话名** | `rnaseq_majus` | Tmux 守护，防止断线。 |
| **基因组文件** | `T_majus.primary_assembly.genome.fa` | 物理锁定，排除 `.raw.fa` 干扰。 |
| **注释文件** | `T_majus.annotation.gtf` | 物理锁定，排除 `.gff3` 和压缩包干扰。 |
| **并行度** | **8 并行** | 针对 232Mb 小基因组实现的极致加速。 |
| **内存上限** | 85GB | 确保服务器系统稳定。 |
| **Index 状态** | 外部载入 (Pre-built) | 直接使用现有 `star_index`，跳过构建步骤。 |


目前状态非常理想，流程正处于从“质控”向“数据清理”过渡的关键节点。
作为 Principal Scientist，我为您分析当前的运行细节：
1. 核心进展解读
样本抓取成功 (34 of 34 ✔)：FASTQC 已经完成了全部 34 个样本的处理。这证明您的 Python 脚本生成的 Samplesheet 物理路径完全正确，Nextflow 已经成功识别并锁定了所有数据。
参考基因组准备中 (0 of 1)：MAKE_TRANSCRIPTS_FASTA 正在运行。由于 T. majus 基因组很小，这一步会非常快，它在为后续的 Salmon 定量准备转录本序列。
进入清理阶段 (TRIMGALORE 0 of 34)：流程刚刚触发了接头修剪（TrimGalore）。这是比对前最后一道“脱壳”工序。
2. 性能与时间评估
TRIMGALORE 阶段：这是高 CPU 消耗阶段。因为您有 32 个核心，Nextflow 应该会同时启动约 16 个 TrimGalore 进程（每个占用 2 核）。
预估耗时：针对 34 个样本，预计在 30 - 50 分钟内完成。
接下来的重头戏：STAR_ALIGN：
一旦 TrimGalore 完成第一个样本，STAR_ALIGN 就会立即启动。
由于我们配置了 maxForks = 8 且基因组极小，STAR 的比对速度会让你感到惊讶。每个样本的比对时间可能被压缩到 5 分钟 以内。
3. 专家级实时监控指标
您可以观察以下变化来确认流程没有“卡死”：
TRIMGALORE 的数字跳动：很快你会看到 [ 3%] 1 of 34，接着是快速增长。
CPU 负载：运行 top 或 htop，你会看到多个 cutadapt 进程（TrimGalore 的内核），这表示数据正在被高强度清理。
硬盘 I/O：你会看到磁盘写速度增加，因为正在生成 .trimmed.fq.gz 临时文件。
4. 潜在风险提示
目前的 MAKE_TRANSCRIPTS_FASTA 如果运行时间超过 10 分钟还没动静，通常是因为正在从 GTF 提取序列，这受限于磁盘读取速度。请保持耐心，目前没有任何报错迹象。
总结结论：
您的项目已经度过了最危险的“路径配置”和“环境初始化”阶段，现在进入了工业化自动生产阶段。如果一切顺利，您将在接下来的 2 小时内 看到所有 34 个样本的比对完成。
建议：继续保持 Tmux 会话挂载，可以暂时去处理其他事务。